# EXP-010 dev-small 확인: RRF dense 가중치 2

KURE-v1 기준(EXP-008), snowflake-ko(EXP-009), snowflake-ko + dense 가중치 2(EXP-010)의 dev-small 지표와 질의별 순위 변화를 본다. 가중치는 dev-full을 보기 전에 2로 고정했고, dev-small에서 크게 무너질 때만 바꾼다(사용자 결정). 판정에 쓰지 않는다.

In [23]:
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, read_run

In [24]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
RUN_FILES = {
    'EXP-008 KURE-v1': 'kure-v1-fixed-512-64-hybrid-cond-filter-law-dev-small.csv',
    'EXP-009 snowflake-ko': 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-dev-small.csv',
    'EXP-010 snowflake-ko w2': 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-w2-dev-small.csv',
}

## 지표

In [25]:
gold = load_gold(load_cfg(), 'dev-small')
ranks = {name: gold_ranks(read_run(RUNS / f), gold, 10) for name, f in RUN_FILES.items()}
print(len(gold), 'queries')
print(pd.DataFrame({name: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                    for name, r in ranks.items()}).round(4))

1039 queries
           EXP-008 KURE-v1  EXP-009 snowflake-ko  EXP-010 snowflake-ko w2
recall@1            0.9105                0.9105                   0.9009
recall@5            0.9808                0.9856                   0.9798
recall@10           0.9875                0.9885                   0.9885


## EXP-009 대비 질의별 이동

In [26]:
b, c = ranks['EXP-009 snowflake-ko'], ranks['EXP-010 snowflake-ko w2']
for k in (1, 5):
    gain = int(((b.isna() | (b > k)) & (c <= k)).sum())
    loss = int(((b <= k) & (c.isna() | (c > k))).sum())
    print(f'top{k}: 진입 {gain}, 이탈 {loss}')

top1: 진입 4, 이탈 14
top5: 진입 0, 이탈 6
